# noise_reachability step 12: a second round of constant-noise search and training (Kaggle, 2 x T4)

Step 12 of `noise_reachability/README.md` (expert iteration, round 2). Step 10's adapter N<t>_0 searches the states
step 10 left weak with fixed `repeat` noises; LoRA adapters R are trained from the base on step 10's recordings plus the
new successes; N0-N2 and R0-R2 are evaluated with sampled noise on fresh episodes.

**Needs step 10's output as input:** Add Input > Your Work > the step 10 notebook, its **last** version (it holds the
adapters and recordings). **Run with Save & Run All (Accelerator: GPU T4 x2, Internet on).** Nothing needs editing. The
wrapper check and the harness run first; a failure of either stops the run.

**This run needs about three sessions.** When step 16 says work remains, add this version's output as input as well
(**Add Input > Your Work**, only the latest step 12 version, and keep step 10's), and Save & Run All again. Step 6 must
then print `RESUMING`.

Download from the Output tab: `step12_results.zip` (results without recordings or adapter weights, which stay in the
version's output for the next session).

## 1. Settings

In [ ]:
import csv, glob, hashlib, json, os, re, shutil, signal, subprocess, sys, time
from pathlib import Path

T_SESSION = time.time()
REPO_URL = "https://github.com/swagatk/efficient_vla.git"
REPO_REF = "SET_AFTER_COMMIT"
assert REPO_REF != "SET_AFTER_COMMIT", "this notebook has no reviewed commit yet"
REPO = "/tmp/efficient_vla"        # code and venv live outside /kaggle/working, so they are not saved as output
VENV = "/tmp/venv"
PY = f"{VENV}/bin/python"

# The protocol of README.md step 12; nothing here is meant to be edited. The analysis refuses anything else.
CHECKPOINT_REPO, CHECKPOINT_REVISION = "lerobot/smolvla_libero", "a24f1f380461b665158d52acd140152c79e8b36d"
TASKS = [0, 6, 7]
N_ACTION_STEPS = 10
SEARCHED = {0: (2, 5, 6, 11), 6: (8, 12, 14), 7: (38,)}       # states round 1 left weak: the round-1 explorer searches them
HELD_OUT = {0: (), 6: (20,), 7: (47,)}       # selected like them, never searched: transfer
CHECK = {0: (8,), 6: (), 7: (33,)}             # round 1's best trained states: kept?
EXPLORER = 0                        # step 10's adapter N<t>_0 explores
SEARCH_IDS = "7000-7049"         # fixed repeat noises, fresh ids
HARNESS_STATE = {0: 8, 6: 8, 7: 33}   # step 10's N<t>_0 episodes re-run, seeds 16-19
HARNESS_SEEDS = "16-19"
TRAIN_SEEDS = [0, 1, 2]
TRAIN_STEPS, BATCH_SIZE = 2000, 16 # step 4's and step 10's settings, unchanged
EVAL_STATES = {0: '2,5,6,8,11', 6: '8,12,14,20', 7: '33,38,47'}
EVAL_SEED_PARTS = {0: ['48-51', '52-55', '56-59', '60-63'], 6: ['48-51', '52-55', '56-59', '60-63', '64-67', '68-71', '72-75', '76-79'], 7: ['48-51', '52-55', '56-59', '60-63', '64-67', '68-71', '72-75', '76-79']}
OUT_NAME = "step12"
OUT_ROOT = f"/kaggle/working/{OUT_NAME}"
ROUND2_DATA = "/tmp/step12"        # round 1's recordings and round 2's, linked per session (not saved as output)
STOP_AT_H = 11.0                   # no new work after this many hours of the session; running jobs are stopped
TRAIN_NEEDS_H = 3.5                # before the timing check: no training run starts with less time left
GB_PER_JOB = 6                     # base policy + one episode in RAM per simulator process

# Commits of the editable installs in the local lerobot_env that produced the existing results.
LEROBOT_COMMIT = "b6ec0060779550c0a157ae34feb89e0cf86012a8"
LIBERO_COMMIT = "8f1084e3132a39270c3a13ebe37270a43ece2a01"
# Round-2 adapters in training order: one task after another per seed. Policies under analyze_step12.py's names:
# N<s> is step 10's adapter N<t>_<s>, R<s> is round 2's R<t>_<s>.
ADAPTERS = [f"R{t}_{s}" for s in TRAIN_SEEDS for t in TASKS]
POLICIES = ['N0', 'N1', 'N2', 'R0', 'R1', 'R2']


def adapter_of(t, policy):
    return f"{policy[0]}{t}_{policy[1:]}"
print(f"step 12 at {REPO_REF}: tasks {TASKS}, searched {SEARCHED}, held out {HELD_OUT}, check {CHECK}, "
      f"explorer N<t>_{EXPLORER}, repeat noise ids {SEARCH_IDS}, adapters {ADAPTERS}, at most "
      f"2236 episodes (12 harness, 400 search, 1824 evaluation), "
      f"stop after {STOP_AT_H} h, output {OUT_ROOT}")


def sh(cmd, check=True, env=None):
    # Capture and print: notebooks do not show output that a subprocess writes directly.
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr).rstrip(), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r


def hours_left():
    return STOP_AT_H - (time.time() - T_SESSION) / 3600

## 2. Hardware

In [ ]:
smi = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader")
gpus = [line.split(", ") for line in smi.stdout.strip().splitlines()]
assert gpus, "No GPU. Set Accelerator to GPU T4 x2."
for name, mem, cap, driver in gpus:
    assert float(cap) >= 7.5, f"{name} (compute capability {cap}) is not supported by the pinned PyTorch build; use GPU T4 x2"
CUDA_DRIVER = float(re.search(r"CUDA Version: ([0-9.]+)", sh("nvidia-smi").stdout).group(1))
mem_gb = int(open("/proc/meminfo").read().split()[1]) / 1e6
avail_gb = int(re.search(r"MemAvailable:\s+(\d+)", open("/proc/meminfo").read()).group(1)) / 1e6
JOBS = max(1, min(2 * len(gpus), int((avail_gb - 4) // GB_PER_JOB), os.cpu_count()))
GPU_IDS = ",".join(str(i) for i in range(len(gpus)))
print(f"{len(gpus)} GPU(s), driver supports CUDA {CUDA_DRIVER}; {os.cpu_count()} CPU cores; "
      f"RAM {mem_gb:.0f} GB ({avail_gb:.0f} GB available) -> {JOBS} evaluations at a time on GPUs {GPU_IDS}")
assert len(gpus) == 2, "Set Accelerator to GPU T4 x2"
assert JOBS >= 4, f"RAM for {JOBS} SmolVLA processes; the notebook runs 4 at once"
assert len(gpus) == 2, "Set Accelerator to GPU T4 x2"

## 3. Code

In [ ]:
if not Path(REPO, ".git").exists():
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch origin && git -C {REPO} checkout {REPO_REF} && git -C {REPO} pull --ff-only origin {REPO_REF}")
sh(f"git -C {REPO} log -1 --format='%H %s'")

## 4. Environment

The package versions of the local `lerobot_env` (as the earlier Kaggle notebooks). Takes several minutes.

In [ ]:
# Headless rendering: MuJoCo needs the NVIDIA EGL vendor file, which some images lack.
icd = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}))
sh("apt-get install -y -qq libegl1 libgles2 libgl1 > /dev/null", check=False)

sh("python -m pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv {VENV} --python 3.12")

req = Path(REPO, "hybrid_diffusion", "requirements-colab.txt").read_text().splitlines()
if CUDA_DRIVER >= 13.0:
    TORCH_BUILD = "cu130 (as pinned)"
    reqs = req
else:
    cu = "cu128" if CUDA_DRIVER >= 12.8 else "cu126"
    assert CUDA_DRIVER >= 12.6, f"driver supports only CUDA {CUDA_DRIVER}; need 12.6 or newer"
    TORCH_BUILD = f"{cu} (driver too old for cu130)"
    torch_pins = [l for l in req if l.split("==")[0] in ("torch", "torchvision")]
    sh(f"uv pip install -q --python {PY} " + " ".join(f"'{p}+{cu}'" for p in torch_pins)
       + f" --index-url https://download.pytorch.org/whl/{cu} --extra-index-url https://pypi.org/simple")
    reqs = [l for l in req if not re.match(r"(torch|torchvision|triton|nvidia-)", l)]
print("PyTorch build:", TORCH_BUILD)
Path("/tmp/requirements.txt").write_text("\n".join(reqs) + "\n")
sh(f"uv pip install -q --python {PY} --no-deps -r /tmp/requirements.txt")
sh(f"uv pip install -q --python {PY} --no-deps 'lerobot @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}'")

LIBERO = os.path.expanduser("~/LIBERO")
if not Path(LIBERO, ".git").exists():
    sh(f"git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git {LIBERO}")
sh(f"git -C {LIBERO} checkout -q {LIBERO_COMMIT}")
Path(LIBERO, "libero", "__init__.py").touch()   # present in the local checkout, missing upstream
sh(f"uv pip install -q --python {PY} --no-deps -e {LIBERO}")

# LIBERO asks for these paths interactively on first import unless the config already exists.
root = f"{LIBERO}/libero/libero"
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
os.makedirs(f"{LIBERO}/libero/datasets", exist_ok=True)
Path(os.path.expanduser("~/.libero/config.yaml")).write_text(
    f"benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n"
    f"datasets: {LIBERO}/libero/datasets\nassets: {root}/assets\n"
)

## 5. Sanity checks, tests and the base checkpoint

In [ ]:
RUN_ENV = dict(os.environ, PATH=f"{VENV}/bin:" + os.environ["PATH"], MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl",
               MPLBACKEND="Agg", WANDB_MODE="disabled", OMP_NUM_THREADS="1", PYTHONPATH=REPO,
               PYTHONFAULTHANDLER="1")
sh(f"""{PY} -c "
import torch, av
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], '| av', av.__version__)
" """, env=RUN_ENV)
egl = sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('EGL context ok')\"", check=False, env=RUN_ENV)
assert "EGL context ok" in egl.stdout, "EGL rendering failed"
sh(f"cd {REPO} && {PY} -m pytest tests/ -q -p no:cacheprovider", env=RUN_ENV)
r = sh(f"""{PY} -c "from huggingface_hub import snapshot_download; print(snapshot_download('{CHECKPOINT_REPO}', revision='{CHECKPOINT_REVISION}'))" """, env=RUN_ENV)
POLICY_PATH = r.stdout.strip().splitlines()[-1]
print("base checkpoint", POLICY_PATH)

## 6. Resume from an earlier version's output

In [ ]:
os.makedirs(OUT_ROOT, exist_ok=True)
previous = [p for p in sorted(glob.glob(f"/kaggle/input/**/{OUT_NAME}", recursive=True))
            if os.path.isdir(p) and os.path.exists(f"{p}/repo_commit.txt")]
assert len(previous) <= 1, f"attach only the latest version's output, found {previous}"
for prev in previous:
    print("RESUMING from", prev)
    shutil.copytree(prev, OUT_ROOT, dirs_exist_ok=True)
if not previous:
    print("FRESH START: no earlier output attached. If this is the second session, stop and add it as input.", flush=True)
head = sh(f"git -C {REPO} log -1 --format='%H'").stdout.strip()
first = Path(OUT_ROOT, "repo_commit.txt")
started_with = first.read_text().split()[0] if first.exists() and first.read_text().strip() else None
if started_with and started_with != head:
    raise RuntimeError(f"the run started at commit {started_with}, this version cloned {head}: set REPO_REF in "
                       f"step 1 to {started_with}")
with open(first, "a") as f:
    f.write(head + "\n")
for d in ("logs", "eval"):
    os.makedirs(f"{OUT_ROOT}/{d}", exist_ok=True)

## 7. Step 10's output: round 1's adapters and recordings

In [ ]:
found = [p for p in sorted(glob.glob("/kaggle/input/**/step10", recursive=True))
         if os.path.exists(f"{p}/runs/N0_0/adapter/run_config.json")]
assert len(found) == 1, f"attach step 10's last version as input (found {found})"
STEP10_IN = found[0]
print("step 10 output:", STEP10_IN, "| its commits:", open(f"{STEP10_IN}/repo_commit.txt").read().split())
for t in TASKS:
    for s in TRAIN_SEEDS:
        assert os.path.exists(f"{STEP10_IN}/runs/N{t}_{s}/adapter/run_config.json"), f"step 10 has no adapter N{t}_{s}"
    eps = [f for f in glob.glob(f"{STEP10_IN}/task_{t}/replay_data/episodes/*.npz") if not f.endswith(".tmp.npz")]
    assert eps, f"step 10 has no recordings for task {t}"
    print(f"task {t}: {len(eps)} round-1 recordings")


def n_adapter(t, s):
    return f"{STEP10_IN}/runs/N{t}_{s}/adapter"

## 8. Helpers: parallel jobs with resume and a deadline

In [ ]:
def run_jobs(jobs, max_parallel, poll_s=30, log_every_s=600):
    """Run {name: (command, gpu)} with at most max_parallel at once; exit code 3 means 'call again' (more
    episodes in a fresh process). No job starts after the deadline and running ones are stopped at it.
    Returns {name: exit code}; a job stopped at the deadline gets None."""
    pending, running, codes, last_log = list(jobs), {}, {}, time.time()
    while pending or running:
        while pending and len(running) < max_parallel and hours_left() > 0:
            name = pending.pop(0)
            cmd, gpu = jobs[name]
            log = open(f"{OUT_ROOT}/logs/{name}.log", "a")
            env = dict(RUN_ENV, CUDA_VISIBLE_DEVICES=str(gpu))
            running[name] = (subprocess.Popen(cmd, shell=True, executable="/bin/bash", env=env, stdout=log,
                                              stderr=subprocess.STDOUT, start_new_session=True), log)
            time.sleep(5)  # staggered EGL start-up (FAIL-006)
        if hours_left() <= 0 and running:
            print(time.strftime("%H:%M"), "deadline: stopping", list(running), flush=True)
            for name, (proc, log) in running.items():
                if proc.poll() is None:
                    os.killpg(proc.pid, signal.SIGTERM)
            time.sleep(20)
            for name, (proc, log) in running.items():
                log.close()
                codes[name] = None
            for name in pending:
                codes[name] = None
            return codes
        if hours_left() <= 0 and not running:
            for name in pending:
                codes[name] = None
            return codes
        for name, (proc, log) in list(running.items()):
            if proc.poll() is None:
                continue
            log.close()
            del running[name]
            if proc.returncode == 3:
                pending.insert(0, name)
                continue
            codes[name] = proc.returncode
            print(time.strftime("%H:%M"), name, "exit", proc.returncode, flush=True)
        if time.time() - last_log > log_every_s:
            last_log = time.time()
            print(time.strftime("%H:%M"), f"{hours_left():.1f} h left; running {list(running)}; "
                  f"{len(pending)} waiting", flush=True)
        time.sleep(poll_s)
    return codes


FIELDS = ["task_id", "seed", "init_state", "success", "steps"]


def read_csv(path):
    return list(csv.DictReader(open(path))) if os.path.exists(path) else []


def write_csv(rows, out):
    os.makedirs(os.path.dirname(out), exist_ok=True)
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows({k: r[k] for k in FIELDS} for r in rows)
    return rows


def n_range(spec):
    a, b = (spec.split("-") + [spec])[:2]
    return int(b) - int(a) + 1


def failed(codes):
    return {k: c for k, c in codes.items() if c not in (0, None)}


FAILURE = None


def fail(message):
    """Keep going to the zip in step 17, which raises this."""
    global FAILURE
    FAILURE = message
    print("FAILED:", message, flush=True)






SEARCH = f"cd {REPO} && {PY} -u noise_reachability/deep_noise_search.py --policy_path {POLICY_PATH} --n_action_steps {N_ACTION_STEPS}"
ANALYZE = f"cd {REPO} && {PY} -S noise_reachability/analyze_step12.py --root {OUT_ROOT}"
MAIN = f"{REPO}/self_filtered_bc/self_filtered_bc.py"


def task_dir(task):
    return f"{OUT_ROOT}/task_{task}"


def sfbc(task):
    return f"cd {REPO} && {PY} -u {MAIN} --task_id {task} --n_action_steps {N_ACTION_STEPS} --policy_path {POLICY_PATH}"


def run_stage(name, jobs, total, max_parallel=4):
    """True when the stage has no work left: a job that failed stops the run, the deadline leaves work for a next version."""
    print(f"{name}: {len(jobs)} of {total} jobs to run; {hours_left():.1f} h left", flush=True)
    if not jobs:
        return True
    if hours_left() <= 0:
        return False
    codes = run_jobs(jobs, max_parallel=max_parallel)
    if failed(codes):
        fail(f"{name} failed: {failed(codes)}")
        return False
    return all(c == 0 for c in codes.values()) and len(codes) == len(jobs)


def run_gate(which):
    if sh(f"{ANALYZE} --gate {which}", check=False, env=RUN_ENV).returncode != 0:
        fail(f"the {which} gate failed: investigate before spending more")
        return False
    return True


def adapter_done(name):
    return Path(OUT_ROOT, "runs", name, "adapter", "run_config.json").exists()


def n_states(spec):
    return len(spec.split(","))


def recordings(folder):
    return [f for f in glob.glob(f"{folder}/episodes/*.npz") if not f.endswith(".tmp.npz")]

## 9. Wrapper check on each task's real model (a few minutes); stops the run if it fails

In [ ]:
CHECK_DONE = True
for t in TASKS:
    r = sh(f"cd {REPO} && {PY} -u noise_reachability/check_step5a.py --task_id {t} --policy_path {POLICY_PATH} --n_action_steps {N_ACTION_STEPS}",
           check=False, env=RUN_ENV)
    if r.returncode != 0:
        fail(f"check_step5a failed on task {t}")
        CHECK_DONE = False

## 10. Harness: step 10's N<t>_0 on four of its own evaluation episodes per task; stops the run if it fails

At least 11 of the 12 episodes must reproduce step 10's rows exactly (success and number of steps).

In [ ]:
HARNESS_DONE = False
if CHECK_DONE and not FAILURE:
    jobs = {}
    for t in TASKS:
        out = f"{task_dir(t)}/harness"
        if len(read_csv(f"{out}/per_episode_lora.csv")) < n_range(HARNESS_SEEDS):
            jobs[f"harness_task{t}"] = (f"{sfbc(t)} --stage eval --eval_policies lora --lora_dir {n_adapter(t, EXPLORER)} "
                                        f"--eval_seeds {HARNESS_SEEDS} --eval_init_states {HARNESS_STATE[t]} "
                                        f"--eval_output_dir {out} --max_episodes_per_process 10", t % 2)
    if run_stage("harness", jobs, len(TASKS)):
        HARNESS_DONE = run_gate("harness")

## 11. Round 2's search: the round-1 explorer with fixed `repeat` noises on the searched states

One process per state; every successful episode is recorded to `task_<t>/search_data/episodes/` for training (failures
are not saved: training uses only successes, and 400 recordings of 30-50 MB would fill the disk).

In [ ]:
SEARCH_DONE = False
if HARNESS_DONE and not FAILURE:
    jobs, k = {}, 0
    for t in TASKS:
        for s in SEARCHED[t]:
            if len(read_csv(f"{task_dir(t)}/search/per_episode_repeat_state{s}.csv")) < n_range(SEARCH_IDS):
                jobs[f"search_task{t}_state{s}"] = (
                    f"{SEARCH} --task_id {t} --family repeat --init_state {s} --noise_ids {SEARCH_IDS} "
                    f"--lora_dir {n_adapter(t, EXPLORER)} --output_dir {task_dir(t)}/search "
                    f"--record_dir {task_dir(t)}/search_data --record_successes_only 1", k % 2)
                k += 1
    SEARCH_DONE = run_stage("search", jobs, sum(len(v) for v in SEARCHED.values()))
for t in TASKS:
    for s in SEARCHED[t]:
        rows = read_csv(f"{task_dir(t)}/search/per_episode_repeat_state{s}.csv")
        print(f"search {t}/{s}: {sum(r['success'] == '1' for r in rows)} of {len(rows)} succeeded")
sh(f"du -sh {OUT_ROOT}; df -h /kaggle/working | tail -1")

## 12. Round 2's training data: step 10's recordings plus the search's, per task

Linked (not copied) under `/tmp/step12/task_<t>/round2_data/episodes/` in every session; training keeps the successes.

In [ ]:
def round2_data(t):
    return f"{ROUND2_DATA}/task_{t}/round2_data"


if SEARCH_DONE and not FAILURE:
    for t in TASKS:             # every search success must have its recording before anything trains on them
        for s in SEARCHED[t]:
            for r in read_csv(f"{task_dir(t)}/search/per_episode_repeat_state{s}.csv"):
                k = int(r["seed"]) - 1000
                name = f"repeat_n{k}_init{s:02d}_seed{r['seed']}.npz"
                if r["success"] == "1" and not os.path.exists(f"{task_dir(t)}/search_data/episodes/{name}"):
                    fail(f"search success {name} of task {t} has no recording")
if SEARCH_DONE and not FAILURE:
    for t in TASKS:
        dest = f"{round2_data(t)}/episodes"
        shutil.rmtree(dest, ignore_errors=True)
        os.makedirs(dest)
        for f in recordings(f"{STEP10_IN}/task_{t}/replay_data") + recordings(f"{task_dir(t)}/search_data"):
            target = f"{dest}/{os.path.basename(f)}"
            assert not os.path.exists(target), f"two recordings named {os.path.basename(f)}"
            os.symlink(f, target)
        print(f"task {t}: {len(recordings(f'{STEP10_IN}/task_{t}/replay_data'))} round-1 and "
              f"{len(recordings(f'{task_dir(t)}/search_data'))} round-2 recordings (successes and failures)")

## 13. Training: R<t>_<s> from the base, tasks 0, 6, 7 x training seeds 0-2

Two adapters at a time, one per T4. A run starts only if the session can finish it (1.25 x the measured time plus 10
minutes); an unfinished run restarts from scratch in the next session.

In [ ]:
TIMING = Path(OUT_ROOT, "train_timing.json")
if SEARCH_DONE and not FAILURE and not TIMING.exists() and hours_left() > 0.5:
    r = sh(f"{sfbc(7)} --stage train --seed 0 --selection success --data_dir {round2_data(7)} --train_steps 20 "
           f"--batch_size {BATCH_SIZE} --log_every 5 --val_every 1000 --output_dir /tmp/step12_train_timing",
           check=False, env=dict(RUN_ENV, CUDA_VISIBLE_DEVICES="0"))
    found = re.findall(r"sec_per_step=([0-9.]+)", r.stdout)
    if r.returncode != 0 or not found:
        fail("the training timing run failed")
    else:
        TIMING.write_text(json.dumps({"sec_per_step": float(found[-1]), "date": time.strftime("%Y-%m-%d %H:%M")}) + "\n")
if TIMING.exists():
    TRAIN_NEEDS_H = 1.25 * TRAIN_STEPS * json.loads(TIMING.read_text())["sec_per_step"] / 3600 + 10 / 60
print(f"a training run starts only with more than {TRAIN_NEEDS_H:.2f} h left")


def train_cmd(name):
    t, s = name[1:].split("_")
    return (f"{sfbc(t)} --stage train --selection success --train_steps {TRAIN_STEPS} --batch_size {BATCH_SIZE} "
            f"--seed {s} --data_dir {round2_data(t)} --output_dir {OUT_ROOT}/runs/{name}")


ready = [n for n in ADAPTERS if not adapter_done(n)] if SEARCH_DONE and not FAILURE else []
print("adapters to train now:", ready or "none", flush=True)
while ready and hours_left() > TRAIN_NEEDS_H and not FAILURE:
    batch, ready = ready[:2], ready[2:]
    for n in batch:
        shutil.rmtree(f"{OUT_ROOT}/runs/{n}", ignore_errors=True)   # an unfinished run restarts from scratch
    codes = run_jobs({f"train_{n}": (train_cmd(n), k) for k, n in enumerate(batch)}, max_parallel=2)
    if failed(codes):
        fail(f"training failed: {failed(codes)}")
    for n in batch:
        lines = open(f"{OUT_ROOT}/logs/train_{n}.log").read().splitlines()
        print(n, [l for l in lines if "training episodes" in l or "training frames" in l or "peak GPU" in l])
DONE_ADAPTERS = [n for n in ADAPTERS if adapter_done(n)]
print("adapters done:", DONE_ADAPTERS)

## 14. Evaluation with sampled noise on fresh episodes: N0-N2 (step 10's) and every R whose adapter is ready

In [ ]:
def eval_out(t, policy, j):
    return f"{task_dir(t)}/eval/{policy}/part{j}"


def lora_of(t, policy):
    return n_adapter(t, policy[1:]) if policy.startswith("N") else f"{OUT_ROOT}/runs/{adapter_of(t, policy)}/adapter"


def eval_cmd(t, policy, j):
    return (f"{sfbc(t)} --stage eval --eval_policies lora --lora_dir {lora_of(t, policy)} "
            f"--eval_seeds {EVAL_SEED_PARTS[t][j]} --eval_init_states {EVAL_STATES[t]} "
            f"--eval_output_dir {eval_out(t, policy, j)} --max_episodes_per_process 10")


def part_size(t, j):
    return n_range(EVAL_SEED_PARTS[t][j]) * n_states(EVAL_STATES[t])


def eval_rows(t, policy):
    return [r for j in range(len(EVAL_SEED_PARTS[t])) for r in read_csv(f"{eval_out(t, policy, j)}/per_episode_lora.csv")]


jobs, k = {}, 0
if HARNESS_DONE and not FAILURE:
    for t in TASKS:
        for policy in POLICIES:
            if policy.startswith("R") and not adapter_done(adapter_of(t, policy)):
                continue
            for j in range(len(EVAL_SEED_PARTS[t])):
                if len(read_csv(f"{eval_out(t, policy, j)}/per_episode_lora.csv")) < part_size(t, j):
                    jobs[f"eval_task{t}_{policy}_part{j}"] = (eval_cmd(t, policy, j), k % 2)
                    k += 1
    run_stage("evaluation", jobs, len(jobs))
for t in TASKS:
    print(f"task {t}:", {p: len(eval_rows(t, p)) for p in POLICIES},
          f"of {sum(part_size(t, j) for j in range(len(EVAL_SEED_PARTS[t])))} each")
ALL_DONE = (len(DONE_ADAPTERS) == len(ADAPTERS) and all(
    len(eval_rows(t, p)) == sum(part_size(t, j) for j in range(len(EVAL_SEED_PARTS[t]))) for t in TASKS for p in POLICIES))

## 15. Analysis (when every episode is done)

In [ ]:
if ALL_DONE and not FAILURE:
    if sh(f"{ANALYZE} --out {OUT_ROOT}/step12_report.txt", check=False, env=RUN_ENV).returncode != 0:
        fail("the analysis failed: the episodes are in the zip")

## 16. Work left

In [ ]:
if not ALL_DONE and not FAILURE:
    print("WORK REMAINS: add this version's output as input (keep step 10's) and Save & Run All again")

## 17. Files for the repo

In [ ]:
sh(f"cd '{OUT_ROOT}' && zip -q -r /kaggle/working/step12_results.zip . -x '*.npz' '*.safetensors' '*.pt'")
sh(f"du -h /kaggle/working/step12_results.zip; du -sh {OUT_ROOT}")
print(f"session time used: {(time.time() - T_SESSION) / 3600:.1f} h")
if FAILURE:
    raise RuntimeError(FAILURE)